*0.4 Deep learning basics*

# GRU (historical context)

**The situation.** The LSTM works but the mobile team needs something smaller and faster for on-device text, and the training budget is tight. In 2014 a simpler gated cell arrived with two gates instead of three and one state instead of two — and about the same accuracy on most tasks.

**GRU (gated recurrent unit).** A *reset* gate decides how much of the old memory to consult when proposing an update; an *update* gate decides how much of the memory to replace. One hidden state, fewer parameters, faster steps.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

In [2]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer

# SST-2: 67k movie-review sentences labelled positive/negative — the standard small sentiment set.
sst2 = load_dataset("stanfordnlp/sst2")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")


def encode(rows, max_length=32):
    encoded = tokenizer(
        rows["sentence"], truncation=True, max_length=max_length, padding="max_length"
    )
    return {"ids": encoded["input_ids"], "label": rows["label"]}


train_rows = sst2["train"].shuffle(seed=0).select(range(8000)).map(encode, batched=True)
val_rows = sst2["validation"].map(encode, batched=True)
train_ids = torch.tensor(train_rows["ids"])
train_labels = torch.tensor(train_rows["label"])
val_ids = torch.tensor(val_rows["ids"])
val_labels = torch.tensor(val_rows["label"])
print(
    "train:",
    tuple(train_ids.shape),
    "| validation:",
    tuple(val_ids.shape),
    "| vocabulary:",
    tokenizer.vocab_size,
)

train: (8000, 32) | validation: (872, 32) | vocabulary: 30522


In [3]:
import time

import torch.nn.functional as F
from torch import nn


class GRUClassifier(nn.Module):
    def __init__(self, vocabulary_size, embedding_size=64, hidden_size=64):
        super().__init__()
        self.embedding = nn.Embedding(vocabulary_size, embedding_size, padding_idx=0)
        self.gru = nn.GRU(embedding_size, hidden_size, batch_first=True)
        self.output = nn.Linear(hidden_size, 2)

    def forward(self, token_ids):
        _, hidden = self.gru(self.embedding(token_ids))  # one state only
        return self.output(hidden[0])


def train(model, epochs=3, lr=2e-3, batch_size=64):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    for epoch in range(1, epochs + 1):
        model.train()
        order = torch.randperm(len(train_ids))
        total_loss = 0.0
        for start in range(0, len(order), batch_size):
            batch = order[start : start + batch_size]
            loss = F.cross_entropy(model(train_ids[batch]), train_labels[batch])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(batch)
        print(
            
                f"epoch {epoch}  train loss {total_loss / len(order):.3f}  val accuracy "
                f"{accuracy(model):.1%}"
            
        )
    return accuracy(model)


def accuracy(model):
    model.eval()
    with torch.no_grad():
        predictions = model(val_ids).argmax(dim=1)
    return (predictions == val_labels).float().mean().item()


torch.manual_seed(0)
started = time.perf_counter()
gru_accuracy = train(GRUClassifier(tokenizer.vocab_size), epochs=4)
gru_seconds = time.perf_counter() - started
print(f"GRU: {gru_accuracy:.1%} in {gru_seconds:.0f} s")


def count_parameters(module):
    total = 0
    for parameter in module.parameters():
        total += parameter.numel()
    return total


print(
    "parameters per cell — GRU:",
    count_parameters(nn.GRU(64, 64)),
    "| LSTM:",
    count_parameters(nn.LSTM(64, 64)),
)
assert gru_accuracy > 0.7

epoch 1  train loss 0.689  val accuracy 49.3%


epoch 2  train loss 0.678  val accuracy 60.1%


epoch 3  train loss 0.543  val accuracy 69.4%


epoch 4  train loss 0.353  val accuracy 72.4%
GRU: 72.4% in 4 s
parameters per cell — GRU: 24960 | LSTM: 33280


**Reading the output.** Accuracy in the same range as the LSTM, with three quarters of the parameters per cell and a faster epoch. For most sequence tasks of that era the two were interchangeable; GRU was the default when compute was the constraint.

```
RNN    1 state, 0 gates      fades
GRU    1 state, 2 gates      remembers, cheaper
LSTM   2 states, 3 gates     remembers, more capacity
```

**The rule to remember.** GRU is the lighter LSTM. Same idea (gated memory), fewer parts. Historical now, for the same reason as the LSTM.

| Use it when | Don't when | Instead use |
|---|---|---|
| a small sequential model where compute is the constraint; legacy code | new text models | transformer |

**Watch out**
- GRU and LSTM return different state shapes (`hidden` vs `(hidden, cell)`); swapping one for the other means changing the forward pass.
- Multi-layer (`num_layers=2`) and bidirectional variants exist for both; parameter counts multiply.
- Neither can be parallelised across positions; that is the ceiling on their speed, not a bug in your code.